# Transformers

A continuación se muestran los modelos transformers creados para predecir la efectividad de los informes.

In [ ]:
# !git clone https://github.com/chuy-zip/PROYECTO2_DS.git

fatal: destination path 'PROYECTO2_DS' already exists and is not an empty directory.


## Preparacion de datos 

Se decidio poner a prueba la solución propuesta de los ganadores de la competencia sobre este conjunto datos.

https://www.kaggle.com/competitions/feedback-prize-effectiveness/writeups/team-hydrogen-team-hydrogen-1st-place-solution

Partieron del enfoque que las diferentes secciones de un ensayo realmente son interdependientes, por ejemplo un buen argumento depende que anteriormente se haya hecho una buena introducción. Para poder mantener este contexto las **piezas de un ensayo y su calificación debería hacer como un todo**. Es por ello que los datos fueron preparados de una forma diferente, este es un ejemplo de un input para el modelo.

> Lead Position Claim Evidence Counterclaim Rebuttal Evidence Counterclaim Concluding Statement [SEP]  [START] Hi, i'm Isaac, i'm going to be writing about how this face on Mars is a natural landform or if there is life on Mars that made it. The story is about how NASA took a picture of Mars and a face was seen on the planet. NASA doesn't know if the landform was created by life on Mars, or if it is just a natural landform. [END]   [START] On my perspective, I think that the face is a natural landform because I dont think that there is any life on Mars. In these next few paragraphs, I'll be talking about how I think that is is a natural landform [END] … more text follows here

**Estructura**
Al principio del todo se indican en orden las secciones que conforman el ensayo. Justo después se muestra el ensayo completo, utilizando las palabras `[SEP]` y `[START]`, para indicar cuando empieza y termina un sección.

Añadido a esto se crea un vector con las puntuaciones de de cada sección usando la codificacion de : 

- Inefectivo = 0
- Adecuado = 1
- Effectivo = 2

De tal manera que al encodificar un ensayo se obtiene 2 valores

1. El texto ensayo completo con los tokens separadores
2. Vector de punteos de cada sección

In [1]:
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, AutoConfig
import torch.nn as nn

torch.cuda.empty_cache()

/home/smaug/Documents/DataScience/PROYECTO2_DS/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [11]:
def format_essay(essay_df):
    """
    Takes all discourses from one essay and formats them into the required input.
    
    Args:
        essay_df: DataFrame containing all discourses for a single essay
        
    Returns:
        formatted_text: String in the required format
        labels: List of effectiveness labels (0=Inadequate, 1=Adequate, 2=Effective)
    """
    # Step 1: Create the discourse type header
    discourse_types = essay_df['discourse_type'].tolist()
    header = ' '.join(discourse_types)
    
    # Step 2: Format each discourse with [START] and [END] markers
    discourse_parts = []
    labels = []
    
    label_map = {'Ineffective': 0, 'Adequate': 1, 'Effective': 2}
    
    for _, row in essay_df.iterrows():
        text = row['discourse_text'].strip()
        discourse_part = f"[START] {text} [END]"
        discourse_parts.append(discourse_part)
        
        # Store the label
        labels.append(label_map[row['discourse_effectiveness']])
    
    # Step 3: Combine everything
    formatted_text = header + " [SEP] " + " ".join(discourse_parts)
    
    return formatted_text, labels


df = pd.read_csv('../data/train.csv')

# Let's format the first essay as an example
first_essay_id = df['essay_id'].iloc[0]
essay_df = df[df['essay_id'] == first_essay_id]
formatted_text, labels = format_essay(essay_df)

print("FORMATTED TEXT:")
print("=" * 80)
print(formatted_text)
print("\n" + "=" * 80)
print(f"\nSCORE VECTOR: {labels}")
print(f"Number of discourses: {len(labels)}")

# Let's also format ALL essays and save them
print("\n" + "=" * 80)
print("Formatting all essays...")

all_formatted_data = []

for essay_id in df['essay_id'].unique():
    essay_df = df[df['essay_id'] == essay_id]
    formatted_text, labels = format_essay(essay_df)
    
    all_formatted_data.append({
        'essay_id': essay_id,
        'formatted_text': formatted_text,
        'labels': labels,
        'num_discourses': len(labels),
        'text_length': len(formatted_text)  # Add character count
    })

# Convert to DataFrame and save
formatted_df = pd.DataFrame(all_formatted_data)

print(f"Formatted {len(formatted_df)} essays!")

# Show statistics
print(f"\nStatistics:")
print(f"  Total essays: {len(formatted_df)}")
print(f"  Avg discourses per essay: {formatted_df['num_discourses'].mean():.1f}")
print(f"  Max discourses in an essay: {formatted_df['num_discourses'].max()}")
print(f"  Min discourses in an essay: {formatted_df['num_discourses'].min()}")
print(f"\n  Text Length (characters):")
print(f"    Average: {formatted_df['text_length'].mean():.0f}")
print(f"    Maximum: {formatted_df['text_length'].max()}")
print(f"    Minimum: {formatted_df['text_length'].min()}")
print(f"    Median: {formatted_df['text_length'].median():.0f}")

# Calculate approximate token length (rough estimate: 1 token ≈ 4 characters)
formatted_df['approx_tokens'] = formatted_df['text_length'] / 4

print(f"\n  Estimated Token Length (approx):")
print(f"    Average: {formatted_df['approx_tokens'].mean():.0f}")
print(f"    Maximum: {formatted_df['approx_tokens'].max():.0f}")
print(f"    Minimum: {formatted_df['approx_tokens'].min():.0f}")
print(f"    Median: {formatted_df['approx_tokens'].median():.0f}")

# Check how many essays exceed common max_length values
print(f"\n  Essays exceeding token limits:")
print(f"    > 512 tokens: {(formatted_df['approx_tokens'] > 512).sum()} ({(formatted_df['approx_tokens'] > 512).mean()*100:.1f}%)")
print(f"    > 1024 tokens: {(formatted_df['approx_tokens'] > 1024).sum()} ({(formatted_df['approx_tokens'] > 1024).mean()*100:.1f}%)")
print(f"    > 2048 tokens: {(formatted_df['approx_tokens'] > 2048).sum()} ({(formatted_df['approx_tokens'] > 2048).mean()*100:.1f}%)")

del formatted_df
del all_formatted_data

FORMATTED TEXT:
Lead Position Claim Evidence Counterclaim Rebuttal Evidence Counterclaim Concluding Statement [SEP] [START] Hi, i'm Isaac, i'm going to be writing about how this face on Mars is a natural landform or if there is life on Mars that made it. The story is about how NASA took a picture of Mars and a face was seen on the planet. NASA doesn't know if the landform was created by life on Mars, or if it is just a natural landform. [END] [START] On my perspective, I think that the face is a natural landform because I dont think that there is any life on Mars. In these next few paragraphs, I'll be talking about how I think that is is a natural landform [END] [START] I think that the face is a natural landform because there is no life on Mars that we have descovered yet [END] [START] If life was on Mars, we would know by now. The reason why I think it is a natural landform because, nobody live on Mars in order to create the figure. It says in paragraph 9, "It's not easy to target Cy

Arriba hay un ejemplo de la codificación de un solo ensayo.

### DataLoader

Para evitar cargar todos los datos en memoria se implemento una clase DataLoader para cargar los datos en baches que seran cargados y liberados cada cierto tiempo. **Cada bache esta compuesto por un solo ensayo y todas sus secciones** justo como se menciono en la sección anterior.

In [3]:
class SimpleEssayDataset(Dataset):
    """
    Minimal Dataset class that PyTorch DataLoader needs.
    Just handles tokenization and organizing the data.
    """
    
    def __init__(self, csv_path, tokenizer_name='microsoft/deberta-v3-large', max_length=1024):
        # Load CSV and group by essay
        df = pd.read_csv(csv_path)
        self.essay_ids = df['essay_id'].unique()
        self.df = df
        self.max_length = max_length
        
        # Initialize tokenizer (use_fast=False to avoid conversion errors)
        self.tokenizer = AutoTokenizer.from_pretrained(tokenizer_name)
        
        # Add special tokens [START] and [END]
        self.tokenizer.add_special_tokens({
            'additional_special_tokens': ['[START]', '[END]']
        })
        
        print(f"Loaded {len(self.essay_ids)} essays")
        print(f"Tokenizer vocabulary size: {len(self.tokenizer)}")
    
    def __len__(self):
        """Returns the number of essays in the dataset"""
        return len(self.essay_ids)
    
    def __getitem__(self, idx):
        """
        Get one essay and return it tokenized.
        
        Returns a dictionary with:
            - essay_id: The essay identifier
            - input_ids: Tokenized text (tensor of token IDs)
            - attention_mask: Mask for padding tokens
            - labels: Effectiveness labels for each discourse
            - formatted_text: The formatted text (for debugging)
        """
        # Get one essay
        essay_id = self.essay_ids[idx]
        essay_df = self.df[self.df['essay_id'] == essay_id]
        
        # Format it
        formatted_text, labels = format_essay(essay_df)
        
        # Tokenize it (convert text to numbers)
        encoding = self.tokenizer(
            formatted_text,
            truncation=True,
            max_length=self.max_length,
            padding='max_length',
            return_tensors='pt'
        )
        
        return {
            'essay_id': essay_id,
            'input_ids': encoding['input_ids'].squeeze(0),  # Remove batch dimension
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'labels': torch.tensor(labels, dtype=torch.long),
            'num_discourses': len(labels),
            'formatted_text': formatted_text  # Keep for debugging
        }

A continuación una pequeña prueba del dataset usando el modelo preentrenado **`deberta-base`**, esto incluye el su *tokenizador* y el *modelo* en si.

In [4]:
# Create dataset
print("Creating dataset...")
dataset = SimpleEssayDataset(
    csv_path='../data/train.csv',
    tokenizer_name='microsoft/deberta-base',
    max_length=2048
)

print(f"\nDataset has {len(dataset)} essays")

# Look at one example
print("\n" + "=" * 80)
print("EXAMPLE 1: Getting a single essay")
print("=" * 80)

sample = dataset[0]

print(f"Essay ID: {sample['essay_id']}")
print(f"Number of discourses: {sample['num_discourses']}")
print(f"Input IDs shape: {sample['input_ids'].shape}")
print(f"Attention mask shape: {sample['attention_mask'].shape}")
print(f"Labels: {sample['labels']}")
print(f"Labels shape: {sample['labels'].shape}")

# Decode the tokens to see the formatted text
decoded_text = dataset.tokenizer.decode(sample['input_ids'], skip_special_tokens=False)
print(f"\nFormatted text (first 600 characters):")
print(decoded_text[:600])

# Create DataLoader
print("\n" + "=" * 80)
print("EXAMPLE 2: Using with DataLoader")
print("=" * 80)

dataloader = DataLoader(
    dataset, 
    batch_size=1,  # Process one essay at a time
    shuffle=True
)

# Get one batch
batch = next(iter(dataloader))

print(f"Batch essay ID: {batch['essay_id']}")
print(f"Batch input_ids shape: {batch['input_ids'].shape}")  # [1, max_length]
print(f"Batch labels shape: {batch['labels'].shape}")  # [1, num_discourses]
print(f"Batch labels: {batch['labels']}")

# Iterate through a few batches
print("\n" + "=" * 80)
print("EXAMPLE 3: Iterating through batches")
print("=" * 80)

for i, batch in enumerate(dataloader):
    if i >= 3:  # Just show first 3
        break
    
    print(f"\nBatch {i+1}:")
    print(f"  Essay ID: {batch['essay_id'][0]}")
    print(f"  Num discourses: {batch['num_discourses'].item()}")
    print(f"  Labels: {batch['labels'][0].tolist()}")

print("\n" + "=" * 80)
print("Dataset is ready to use for training!")
print("=" * 80)

Creating dataset...
Loaded 4191 essays
Tokenizer vocabulary size: 50267

Dataset has 4191 essays

EXAMPLE 1: Getting a single essay
Essay ID: 007ACE74B050
Number of discourses: 9
Input IDs shape: torch.Size([2048])
Attention mask shape: torch.Size([2048])
Labels: tensor([1, 1, 1, 1, 1, 0, 1, 1, 1])
Labels shape: torch.Size([9])

Formatted text (first 600 characters):
[CLS]Lead Position Claim Evidence Counterclaim Rebuttal Evidence Counterclaim Concluding Statement [SEP] [START] Hi, i'm Isaac, i'm going to be writing about how this face on Mars is a natural landform or if there is life on Mars that made it. The story is about how NASA took a picture of Mars and a face was seen on the planet. NASA doesn't know if the landform was created by life on Mars, or if it is just a natural landform. [END] [START] On my perspective, I think that the face is a natural landform because I dont think that there is any life on Mars. In these next few paragraphs, I'll be ta

EXAMPLE 2: Using with DataLo

## Modelo

Para el modelo, se estará utilizando un modelo pre-entrenado al que se estara realizando algunas modificaciones para que pueda dar el resultado correcto. 

La idea principal es entrenarlo con **1 ensayo a la vez y prediciendo el puntaje para cada sección** , es decir:

`1 input -> N ouputs`

Donde N es el conjunto de secciones que conforman el input.

In [5]:
class EssayGroupModel(nn.Module):
    """
    Model that processes full essays and predicts effectiveness for each discourse.
    
    How it works:
    1. Takes ONE full essay as input
    2. Passes it through the backbone (DeBERTa/RoBERTa)
    3. Pools embeddings between [START] and [END] tokens for each discourse
    4. Predicts effectiveness for each discourse independently
    """
    
    def __init__(
        self,
        model_name='microsoft/deberta-base',
        num_labels=3,  # Inadequate, Adequate, Effective
        dropout=0.1
    ):
        super().__init__()
        
        # Load pretrained backbone
        print(f"Loading pretrained model: {model_name}")
        self.backbone = AutoModel.from_pretrained(model_name)
        self.config = self.backbone.config
        self.hidden_size = self.config.hidden_size
        
        # Dropout for regularization
        self.dropout = nn.Dropout(dropout)
        
        # Final classification layer
        self.classifier = nn.Linear(self.hidden_size, num_labels)
        
        print(f"Model initialized with hidden_size={self.hidden_size}")
    
    def resize_token_embeddings(self, new_num_tokens):
        """
        Call this after adding special tokens to the tokenizer.
        This resizes the model's embedding layer to accommodate new tokens.
        """
        self.backbone.resize_token_embeddings(new_num_tokens)
        print(f"Resized token embeddings to {new_num_tokens}")
    
    def find_start_end_positions(self, input_ids, start_token_id, end_token_id):
        """
        Find all [START] and [END] token positions in the input.
        
        Args:
            input_ids: [seq_len] tensor of token IDs
            start_token_id: ID of [START] token
            end_token_id: ID of [END] token
            
        Returns:
            start_positions: List of start token positions
            end_positions: List of end token positions
        """
        start_positions = []
        end_positions = []
        
        for i, token_id in enumerate(input_ids):
            if token_id == start_token_id:
                start_positions.append(i)
            elif token_id == end_token_id:
                end_positions.append(i)
        
        return start_positions, end_positions
    
    def pool_discourse_embeddings(self, sequence_output, start_positions, end_positions):
        """
        Pool embeddings between [START] and [END] tokens for each discourse.
        
        Args:
            sequence_output: [seq_len, hidden_size] - output from backbone
            start_positions: List of start token positions
            end_positions: List of end token positions
            
        Returns:
            pooled_embeddings: [num_discourses, hidden_size]
        """
        num_discourses = len(start_positions)
        pooled_embeddings = []
        
        for i in range(num_discourses):
            start_idx = start_positions[i]
            end_idx = end_positions[i]
            
            # Extract tokens between START and END (exclusive of the markers)
            # discourse_embeddings shape: [num_tokens_in_discourse, hidden_size]
            discourse_embeddings = sequence_output[start_idx + 1:end_idx]
            
            if len(discourse_embeddings) == 0:
                # If empty (shouldn't happen), use the START token
                pooled = sequence_output[start_idx]
            else:
                # Mean pooling
                pooled = discourse_embeddings.mean(dim=0)
            
            pooled_embeddings.append(pooled)
        
        # Stack into tensor: [num_discourses, hidden_size]
        pooled_embeddings = torch.stack(pooled_embeddings, dim=0)
        
        return pooled_embeddings
    
    def forward(self, input_ids, attention_mask, start_token_id, end_token_id, labels=None):
        """
        Forward pass.
        
        Args:
            input_ids: [batch_size=1, seq_len] - tokenized essay
            attention_mask: [batch_size=1, seq_len] - attention mask
            start_token_id: int - ID of [START] token
            end_token_id: int - ID of [END] token
            labels: [num_discourses] - effectiveness labels (optional, for training)
            
        Returns:
            Dictionary with:
                - loss: scalar (if labels provided)
                - logits: [num_discourses, 3] - predictions for each discourse
                - probabilities: [num_discourses, 3] - softmax probabilities
        """
        # Pass through backbone
        outputs = self.backbone(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        
        # Get sequence output: [batch_size=1, seq_len, hidden_size]
        sequence_output = outputs.last_hidden_state
        
        # Remove batch dimension since we process one essay at a time
        sequence_output = sequence_output.squeeze(0)  # [seq_len, hidden_size]
        input_ids_1d = input_ids.squeeze(0)  # [seq_len]
        
        # Find [START] and [END] positions
        start_positions, end_positions = self.find_start_end_positions(
            input_ids_1d, start_token_id, end_token_id
        )
        
        # Pool embeddings for each discourse
        pooled_embeddings = self.pool_discourse_embeddings(
            sequence_output, start_positions, end_positions
        )  # [num_discourses, hidden_size]
        
        # Apply dropout
        pooled_embeddings = self.dropout(pooled_embeddings)
        
        # Get logits for each discourse
        logits = self.classifier(pooled_embeddings)  # [num_discourses, 3]
        
        # Calculate probabilities
        probabilities = torch.softmax(logits, dim=-1)
        
        # Calculate loss if labels provided
        loss = None
        if labels is not None:
            loss_fct = nn.CrossEntropyLoss()
            loss = loss_fct(logits, labels)
        
        return {
            'loss': loss,
            'logits': logits,
            'probabilities': probabilities,
            'num_discourses': len(start_positions)
        }




**Probando el modelo**

Para verificar que todas el modelo funciona bien, se hizo una pequeña aislada con datos de prueba para comprobar que los inputs y outputs eran los requeridos.

In [6]:
# Initialize model
model = EssayGroupModel(
    model_name='microsoft/deberta-base',
    num_labels=3,
    dropout=0.1
)

# Initialize tokenizer and add special tokens
tokenizer = AutoTokenizer.from_pretrained('microsoft/deberta-base', use_fast=False)
tokenizer.add_special_tokens({
    'additional_special_tokens': ['[START]', '[END]']
})

# IMPORTANT: Resize model embeddings after adding tokens
model.resize_token_embeddings(len(tokenizer))

print(f"\nModel Summary:")
print(f"  Total parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"  Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

# Create a dummy batch to test
dummy_text = "Lead Position [SEP] [START] This is the first discourse. [END] [START] This is the second discourse. [END]"

encoding = tokenizer(
    dummy_text,
    return_tensors='pt',
    max_length=512,
    padding='max_length',
    truncation=True
)

dummy_labels = torch.tensor([1, 2])  # Adequate, Effective

# Test forward pass
print("\nTesting forward pass...")
outputs = model(
    input_ids=encoding['input_ids'],
    attention_mask=encoding['attention_mask'],
    start_token_id=tokenizer.convert_tokens_to_ids('[START]'),
    end_token_id=tokenizer.convert_tokens_to_ids('[END]'),
    labels=dummy_labels
)

print(f"  Loss: {outputs['loss'].item():.4f}")
print(f"  Logits shape: {outputs['logits'].shape}")
print(f"  Probabilities shape: {outputs['probabilities'].shape}")
print(f"  Number of discourses found: {outputs['num_discourses']}")
print(f"  Predictions:\n{outputs['probabilities']}")

print("\n✅ Model is ready to train!")

Loading pretrained model: microsoft/deberta-base
Model initialized with hidden_size=768


The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


Resized token embeddings to 50267

Model Summary:
  Total parameters: 138,605,571
  Trainable parameters: 138,605,571

Testing forward pass...
  Loss: 0.9687
  Logits shape: torch.Size([2, 3])
  Probabilities shape: torch.Size([2, 3])
  Number of discourses found: 2
  Predictions:
tensor([[0.2585, 0.5441, 0.1973],
        [0.2399, 0.4953, 0.2648]], grad_fn=<SoftmaxBackward0>)

✅ Model is ready to train!


Como se puede observar el modelo devolvio una matriz de predicciones

- Cada fila contiene las predicciones para cada sección.
- Cade valor representa una probabiliad que es sección pertenezca a cierta categoria (efectivo, inadecuado, adecuado).

Ejemplo:

```python
[[0.2585, 0.5441, 0.1973],
[0.2399, 0.4953, 0.2648]]
```

La seccion 1 tine 22,85 probabilidades de ser inadecuada, 54% de ser adecuada y 19% de ser effectiva

### Entrenamiento
(Gerax te dejo esto por si te sirve, son solo boradores de las funciones de entrenamiento y evaluación)

In [ ]:
def train_step(model, batch, optimizer, device, tokenizer):
    """
    Single training step for one essay.
    
    Args:
        model: The EssayGroupModel
        batch: Dictionary from DataLoader
        optimizer: PyTorch optimizer
        device: 'cuda' or 'cpu'
        tokenizer: The tokenizer (to get special token IDs)
        
    Returns:
        loss: float - loss value
        num_discourses: int - number of discourses in this essay
    """
    model.train()
    
    # Move to device
    input_ids = batch['input_ids'].to(device)
    attention_mask = batch['attention_mask'].to(device)
    labels = batch['labels'].squeeze(0).to(device)  # Remove batch dimension
    
    # Get special token IDs
    start_token_id = tokenizer.convert_tokens_to_ids('[START]')
    end_token_id = tokenizer.convert_tokens_to_ids('[END]')
    
    # Forward pass
    outputs = model(
        input_ids=input_ids,
        attention_mask=attention_mask,
        start_token_id=start_token_id,
        end_token_id=end_token_id,
        labels=labels
    )
    
    loss = outputs['loss']
    
    # Backward pass
    optimizer.zero_grad()
    loss.backward()
    
    # Gradient clipping (prevents exploding gradients)
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    
    optimizer.step()
    
    return loss.item(), outputs['num_discourses']


### Evaluación

In [ ]:
def eval_step(model, batch, device, tokenizer):
    """
    Single evaluation step for one essay.
    
    Returns:
        loss: float
        predictions: numpy array [num_discourses, 3]
        labels: numpy array [num_discourses]
    """
    model.eval()
    
    with torch.no_grad():
        # Move to device
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].squeeze(0).to(device)
        
        # Get special token IDs
        start_token_id = tokenizer.convert_tokens_to_ids('[START]')
        end_token_id = tokenizer.convert_tokens_to_ids('[END]')
        
        # Forward pass
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            start_token_id=start_token_id,
            end_token_id=end_token_id,
            labels=labels
        )
        
        loss = outputs['loss'].item()
        predictions = outputs['probabilities'].cpu().numpy()
        labels_np = labels.cpu().numpy()
        
        return loss, predictions, labels_np